# learned

> The review queue for `<cfg>/learned`: what Dhrona proposed waits on a branch until a person accepts or rejects it.

In [ ]:
#| default_exp learned

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

Dhrona is the optional package (the `dhrona` extra) that learns from past sessions. The learned folder is `<cfg>/learned`, a git repository. Dhrona's nightly batch never changes the files that Ramabana briefs from. It commits each candidate it keeps on its own `pending/<id>` branch of that repository. The briefing and skill discovery read only the folder's files, which are its HEAD. A pending candidate therefore has no effect until a person accepts it. `--no-learned` leaves out the folder whatever it holds.

`pending` lists what waits, `diff` shows one candidate's change, `accept` moves it onto HEAD, and `reject` drops it. Each of `diff`, `accept` and `reject` returns a dict with `ok` and `id`, and with `error` when `ok` is False. A client application can display the returned dict directly. `accept` and `reject` take the lock that a Dhrona batch holds, and refuse while a batch runs. `diff` takes no lock. `accept` refuses only when the learned folder has an uncommitted change to a file that the candidate also changes. A hand edit to another file stays as it is, unstaged. Dhrona keeps each lesson as an entry in a dated file, `dhrona-<date>.md`. Two lessons on the same day give two candidates that change the same file, and the newer one holds both entries. `pending` lists only the newest, with the older ids in `includes`. `accept` on the newest deletes the older candidates. `reject` on the newest lists the older ones again. `accept` refuses a candidate that would take `AGENTS.md` past `LEARNED_MAX`. `LEARNED_MAX` is the 2,000 characters that Dhrona allows one candidate. Accepted rounds add up. `accept` keeps HEAD within the limit too.

`compact` asks Dhrona for a shorter `AGENTS.md` when accepted lessons crowd it. Dhrona's editor model merges and shortens the paragraphs that Dhrona proposed. It keeps every paragraph that a person wrote byte for byte. It commits the result as one more pending candidate. `diff`, `accept` and `reject` act on that candidate like any other.

`compact(folder, id)` also folds in pending candidate `id`. This is the case that `accept` refuses for size. The refusal returns `length` and names `/learned compact ID`. Accepting that compaction accepts the candidate with it.

`compact` returns `{ok, id, before, after}`, where `before` and `after` are the file's length in characters. On failure it returns `{ok: False, error}`. It needs the `dhrona` extra and says so when Dhrona is missing. It runs one model turn and takes as long as that turn.

In [ ]:
#| export
import fcntl, json, os, re, subprocess, sys, tempfile
from contextlib import contextmanager
from pathlib import Path
from fastcore.script import call_parse
from fastcore.xtras import atomic_save
from ramabana.setup import set_setting, setting

LEARNED = 'learned'   #: the folder under the config dir where Dhrona keeps what it learned
PENDING = 'pending/'  #: the branch prefix a candidate waits under until a person accepts it
LEARNED_MAX = 2000    #: characters of `AGENTS.md` Dhrona lets a candidate carry; `accept` keeps HEAD within it too
_ID = re.compile(r'\d{8}-\d{6}-[0-9a-f]{6}')
_AS = ('-c', 'user.name=Ramabana', '-c', 'user.email=ramabana@localhost')
DATED = re.compile(r'dhrona-\d{4}-\d{2}-\d{2}\.md')   #: the dated lesson files Dhrona keeps in the learned folder, one per day
INCLUDES = 'Includes: '   #: the trailer naming the candidate a compaction folded in; accepting it accepts that one too

SETTINGS = 'settings.json'   #: where 0.2.44 kept `learning`, in each config folder; read once to move it to config.toml

def _legacy(cfg):
    "`learning` from `<cfg>/settings.json`: True, False, or None when it never said."
    try: v = json.loads((Path(cfg).expanduser()/SETTINGS).read_text()).get('learning')
    except (OSError, ValueError, AttributeError): return None
    return None if v is None else v != 'off'

def enabled(cfg='~/.config/ramabana'):
    "Whether learning is on. Reads `learning` in this machine's config.toml. When only `<cfg>/settings.json` holds a value, the first read moves it into config.toml. On unless set off."
    if (v := setting('learning')) is not None: return v
    if (old := _legacy(cfg)) is None: return True
    try: set_setting('learning', old)
    except (OSError, ValueError): pass   # a read-only home, or a config.toml that is not TOML, keeps answering from settings.json
    return old

def set_enabled(cfg='~/.config/ramabana', on=True):
    "Turn learning on or off in this machine's config.toml and keep its comments. `cfg` is ignored, and `cfg`'s settings.json is never written. Raises `ValueError` for a config.toml that is not TOML."
    return set_setting('learning', bool(on))

def learned_folder(cfg='~/.config/ramabana'):
    "`<cfg>/learned`, whether or not a session runs with it."
    return Path(cfg).expanduser()/LEARNED

def _git(d, *a, check=True): return subprocess.run(['git', '-C', str(d), *a], stdin=subprocess.DEVNULL, capture_output=True, text=True, check=check)

@contextmanager
def _locked(folder):
    "Hold the lock a Dhrona batch takes on `folder`, beside it; yields False while a batch holds it."
    folder = Path(folder)
    folder.parent.mkdir(parents=True, exist_ok=True)
    with open(folder.parent/f'.{folder.name}.lock', 'w') as f:
        try: fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError: yield False; return
        yield True

STALE = 'it no longer applies: the learned folder changed after it was proposed; reject it'   #: why a candidate cannot be accepted any more

def _paths(folder, *a):
    "The paths a git listing command prints for `folder`."
    return [p for p in _git(folder, *a, '-z').stdout.split('\0') if p]

def _picked(folder, ref):
    "The tree that picking `ref` onto HEAD makes, or None when it conflicts."
    r = _git(folder, 'merge-tree', '--write-tree', f'--merge-base={ref}^', 'HEAD', ref, check=False)
    return None if r.returncode else r.stdout.split()[0]

def _stale(folder, ref):
    "`STALE` when `ref` was proposed on an older HEAD and no longer picks cleanly onto it, else None."
    if _git(folder, 'rev-parse', f'{ref}^').stdout == _git(folder, 'rev-parse', 'HEAD').stdout: return None
    return None if _picked(folder, ref) else STALE

def restack(folder):
    "Move each pending candidate proposed on an older HEAD onto HEAD where it picks cleanly; returns `(moved, stale)` ids."
    folder, moved, stale = Path(folder), [], []
    head = _git(folder, 'rev-parse', 'HEAD').stdout.strip()
    for p in _candidates(folder):
        ref = PENDING + p['id']
        if _git(folder, 'rev-parse', f'{ref}^').stdout.strip() == head: continue
        if (tree := _picked(folder, ref)) is None: stale.append(p['id']); continue
        an, ae, ad, msg = _git(folder, 'log', '-1', '--format=%an%x00%ae%x00%ad%x00%B', ref).stdout.split('\0', 3)
        env = os.environ | dict(GIT_AUTHOR_NAME=an, GIT_AUTHOR_EMAIL=ae, GIT_AUTHOR_DATE=ad)
        sha = subprocess.run(['git', '-C', str(folder), *_AS, 'commit-tree', tree, '-p', head, '-m', msg.strip()], stdin=subprocess.DEVNULL, capture_output=True, text=True, check=True, env=env).stdout.strip()
        _git(folder, 'update-ref', f'refs/heads/{ref}', sha); moved.append(p['id'])
    return moved, stale

def _candidates(folder):
    "Every pending branch, oldest first, as `{id, at, subject, files, stale}`."
    folder = Path(folder)
    if not (folder/'.git').exists(): return []
    out = _git(folder, 'for-each-ref', '--sort=authordate', '--format=%(refname:short)%00%(authordate:unix)%00%(subject)',
               f'refs/heads/{PENDING}').stdout
    rows = []
    for l in out.splitlines():
        ref, at, subj = l.split('\0')
        rows.append(dict(id=ref.removeprefix(PENDING), at=int(at), subject=subj,
                         files=_git(folder, 'diff', '--name-only', f'{ref}^', ref).stdout.split(), stale=_stale(folder, ref)))
    return rows

def _dated(p): return p['files'][0] if len(p['files']) == 1 and DATED.fullmatch(p['files'][0]) else None

def _includes(folder, rows):
    "`{id: older ids}`: for a candidate that changes one dated file, the older candidates that change that same file and whose text it starts with. Their entries are in it."
    def text(p, name): return _git(folder, 'show', f"{PENDING}{p['id']}:{name}").stdout
    out = {}
    for new in rows:
        if (name := _dated(new)) is None: continue
        t = text(new, name)
        if (old := [o['id'] for o in rows if o['id'] < new['id'] and _dated(o) == name and t.startswith(text(o, name))]): out[new['id']] = old
    return out

def pending(folder):
    "Each candidate waiting for review, oldest first, as `{id, at, subject, files, stale, includes}`; `at` is when it was proposed, in epoch seconds, and `stale` why it cannot be accepted or None. A candidate that holds the entries of older candidates for the same dated file lists them in `includes`, and they are left out."
    rows = _candidates(folder)
    inc = _includes(folder, rows)
    hidden = {i for ids in inc.values() for i in ids}
    return [dict(p, includes=inc.get(p['id'], [])) for p in rows if p['id'] not in hidden]

def _problem(folder, id):
    "Why `id` cannot be acted on in `folder`, or None."
    if not _ID.fullmatch(str(id or '')): return f'{id!r} is not a pending id; `pending` lists them'
    if not (Path(folder)/'.git').exists(): return f'{folder} holds nothing Dhrona proposed'
    if _git(folder, 'rev-parse', '--verify', '-q', f'refs/heads/{PENDING}{id}', check=False).returncode: return f'nothing is pending as {id}'

def diff(folder, id):
    "What accepting pending candidate `id` would change, as `{ok, id, diff}`."
    if (e := _problem(folder, id)): return dict(ok=False, id=id, error=e)
    return dict(ok=True, id=id, diff=_git(folder, 'diff', f'{PENDING}{id}^', f'{PENDING}{id}').stdout)

def accept(folder, id):
    "Move pending candidate `id` onto HEAD, then restack the other candidates onto it. The next briefing reads HEAD. Returns `{ok, id, commit, moved, stale}`."
    folder = Path(folder)
    if (e := _problem(folder, id)): return dict(ok=False, id=id, error=e)
    with _locked(folder) as held:
        if not held: return dict(ok=False, id=id, error='a Dhrona batch is running on the learned folder; accept it when the batch ends')
        ref = f'{PENDING}{id}'
        dirty = {*_paths(folder, 'diff', '--name-only', 'HEAD'), *_paths(folder, 'ls-files', '--others', '--exclude-standard')}
        if clash := sorted(dirty & set(_paths(folder, 'diff', '--name-only', f'{ref}^', ref))):
            return dict(ok=False, id=id, error=f"{folder} has uncommitted changes to {', '.join(clash)}, which this candidate changes; commit or discard them first")
        older = _includes(folder, _candidates(folder)).get(id, [])
        wt = Path(tempfile.mkdtemp())/'wt'   # pick here first; a conflict never reaches the files a session briefs from
        try:
            _git(folder, 'worktree', 'add', '-q', '--detach', str(wt), 'HEAD')
            r = _git(wt, *_AS, 'cherry-pick', f'{PENDING}{id}', check=False)
            if r.returncode and 'empty' not in r.stdout + r.stderr: return dict(ok=False, id=id, error=STALE)
            sha = None if r.returncode else _git(wt, 'rev-parse', 'HEAD').stdout.strip()
            if sha and (n := len((wt/'AGENTS.md').read_text()) if (wt/'AGENTS.md').exists() else 0) > LEARNED_MAX:
                return dict(ok=False, id=id, length=n, error=f'accepting it would make AGENTS.md {n:,} characters, past the {LEARNED_MAX:,} Dhrona allows; '
                                                             f'reject it, run `/learned compact {id}` to fold it into a shorter AGENTS.md, or trim AGENTS.md and commit that first')
            if sha: _git(folder, 'merge', '-q', '--ff-only', sha)
        finally: _git(folder, 'worktree', 'remove', '--force', str(wt), check=False)
        body = _git(folder, 'log', '-1', '--format=%B', f'{PENDING}{id}').stdout
        folded = [l.removeprefix(INCLUDES).strip() for l in body.splitlines() if l.startswith(INCLUDES)]
        for f in [id, *folded, *older]: _git(folder, 'branch', '-D', f'{PENDING}{f}', check=False)
        moved, stale = restack(folder)
        return dict(ok=True, id=id, commit=sha[:7] if sha else None, moved=moved, stale=stale)   # commit None: HEAD already held the change

def compact(folder, id=None, chat=None):
    "Ask Dhrona for a shorter `AGENTS.md` as a pending candidate, folding in pending `id` if given: `{ok, id, before, after}` in characters, or `{ok: False, error}`."
    try: from dhrona.improve import compact_learned
    except ImportError: return dict(ok=False, error='compacting needs dhrona 0.1.12 or later: uv add "ramabana[dhrona]"')
    if id and (e := _problem(folder, id)): return dict(ok=False, error=e)
    r = compact_learned(folder, chat=chat, onto=id or None)
    return dict(ok=True, id=r['pending'].removeprefix(PENDING), before=r['before'], after=r['after']) if r['ok'] else r

def learned_command(folder, arg=''):
    "`/learned` and `ramabana-learned` as text: `list` by default, `compact`, `on`, `off` or `status`, or `diff`, `accept` or `reject` with an id."
    cmd, _, id = (arg or 'list').strip().partition(' ')
    if cmd in ('on', 'off'):
        try: set_enabled(Path(folder).parent, cmd == 'on')
        except ValueError as e: return str(e)
    if cmd in ('on', 'off', 'status'): return f"learning is {'on' if enabled(Path(folder).parent) else 'off'}"
    if cmd == 'list':
        rows = pending(folder)
        return '\n'.join(f"{p['id']}  {', '.join(p['files'])}  {p['subject']}" + (f"  (stale: {p['stale']})" if p['stale'] else '') + (f"  (includes the entries of {', '.join(p['includes'])}; accepting it drops them)" if p['includes'] else '') for p in rows) or 'nothing is pending'
    if cmd == 'compact':
        r = compact(folder, id.strip())
        return f"compaction pending as {r['id']}: AGENTS.md {r['before']:,} → {r['after']:,} characters; /learned diff {r['id']} shows it" if r['ok'] else r['error']
    act = dict(diff=diff, accept=accept, reject=reject).get(cmd)
    if act is None: return f'{cmd!r}: say /learned, /learned compact [ID], /learned on|off|status, or /learned diff|accept|reject ID'
    r = act(folder, id.strip())
    if not r['ok']: return r['error']
    if cmd == 'diff': return r['diff'] or '(no change)'
    out = f"{cmd}ed {r['id']}" + (f" as {r['commit']}" if r.get('commit') else '')
    if r.get('moved'): out += f"; moved {', '.join(r['moved'])} onto it"
    if r.get('stale'): out += f"; {', '.join(r['stale'])} no longer apply: reject them"
    return out

def reject(folder, id):
    "Drop pending candidate `id`; `{ok, id}`."
    if (e := _problem(folder, id)): return dict(ok=False, id=id, error=e)
    with _locked(folder) as held:
        if not held: return dict(ok=False, id=id, error='a Dhrona batch is running on the learned folder; reject it when the batch ends')
        _git(folder, 'branch', '-D', f'{PENDING}{id}')
        return dict(ok=True, id=id)

In [ ]:
# git gets end of input, and a line waiting on the session's terminal stays there
import os, pty, subprocess, sys
m, s = pty.openpty(); os.write(m, b'x\n')
probe = "import os, select, tempfile; from ramabana.learned import _git; print(_git(tempfile.gettempdir(), 'hash-object', '--stdin').stdout.strip(), os.read(0, 9) if select.select([0], [], [], 1)[0] else b'')"
got = subprocess.run([sys.executable, '-c', probe], stdin=s, capture_output=True, text=True, timeout=60).stdout.split()
os.close(s); os.close(m)
test_eq(got, ['e69de29bb2d1d6434b8b29ae775ad8c2e48c5391', "b'x\\n'"])

`accept` first applies a candidate in a throwaway git worktree. A conflict never leaves markers in the files that a running session briefs from. HEAD moves only when the pick is clean. After HEAD moves, `restack` moves every other candidate proposed on an older HEAD onto the new HEAD where it picks cleanly. `accept` returns those ids as `moved`. A candidate that no longer picks cleanly is listed as `stale`. An example is a candidate that edits a line that a compaction rewrote. `accept` returns those ids as `stale`, and `pending` gives the reason in each candidate's `stale` field. A host can show the reason beside the candidate. Ids sort by when Dhrona proposed them, and moving a candidate does not change its id. A candidate whose change HEAD already holds is accepted with `commit` set to `None`. The commit keeps Dhrona as its author. Its commit time is the moment of acceptance, and Dhrona's 30-day expiry clock starts there.

In [ ]:
import os, json, time
def git(d, *a): return subprocess.run(['git', '-C', str(d), *a], check=True, capture_output=True, text=True).stdout.strip()
def propose(folder, files, id):
    "What Dhrona's `keep` does: a commit on `pending/<id>` off HEAD, the folder's files untouched."
    wt = Path(tempfile.mkdtemp())/'wt'
    git(folder, 'worktree', 'add', '-q', '-b', PENDING + id, str(wt), 'HEAD')
    for rel, text in files.items(): (wt/rel).parent.mkdir(parents=True, exist_ok=True); (wt/rel).write_text(text)
    git(wt, 'add', '-A'); git(wt, '-c', 'user.name=Dhrona', '-c', 'user.email=d@x', 'commit', '-qm', f'Lesson from a replay: {id}')
    git(folder, 'worktree', 'remove', '--force', str(wt))

with tempfile.TemporaryDirectory() as d:
    f = Path(d)/'learned'; f.mkdir()
    (f/'AGENTS.md').write_text('A rule a person wrote.\n')
    git(f, 'init', '-q'); git(f, 'add', '-A'); git(f, '-c', 'user.name=t', '-c', 'user.email=t@x', 'commit', '-qm', 'first')
    a, b, c = '20261019-010101-aaaaaa', '20261019-020202-bbbbbb', '20261019-030303-cccccc'
    propose(f, {'skills/sizes/SKILL.md': '---\nname: sizes\ndescription: x\n---\nuse ls\n'}, a)
    propose(f, {'AGENTS.md': 'Something else entirely.\n'}, b)
    propose(f, {'AGENTS.md': 'A third view.\n'}, c)
    test_eq([(p['id'], p['files']) for p in pending(f)], [(a, ['skills/sizes/SKILL.md']), (b, ['AGENTS.md']), (c, ['AGENTS.md'])])
    test_eq((f/'AGENTS.md').read_text(), 'A rule a person wrote.\n')
    assert not (f/'skills').exists()
    assert '+use ls' in diff(f, a)['diff']
    r = accept(f, a)
    test_eq(r['moved'], [b, c])
    assert r['ok'] and r['commit'] and (f/'skills'/'sizes'/'SKILL.md').exists() and a not in [p['id'] for p in pending(f)]
    test_eq(git(f, 'log', '-1', '--format=%an'), 'Dhrona')
    test_eq(accept(f, b)['ok'], True)
    test_eq([p['stale'] for p in pending(f)], [STALE])     # `c` rewrites the line `b` changed
    head, text = git(f, 'rev-parse', 'HEAD'), (f/'AGENTS.md').read_text()
    r = accept(f, c)
    assert not r['ok'], r
    test_eq((git(f, 'rev-parse', 'HEAD'), (f/'AGENTS.md').read_text()), (head, text))
    assert '<<<<' not in text and git(f, 'status', '--porcelain') == ''
    test_eq(reject(f, c), dict(ok=True, id=c))
    test_eq(pending(f), [])
    test_eq(accept(f, c)['ok'], False)
    test_eq(accept(f, '--force')['ok'], False)             # not an id, never an option
    propose(f, {'AGENTS.md': 'Hand edit pending.\n'}, a)
    (f/'AGENTS.md').write_text('an uncommitted edit\n')
    test_eq(accept(f, a)['ok'], False)                      # a hand edit blocks it
    git(f, 'checkout', '-q', '--', '.')
    with _locked(f) as held:                                # a batch holds it
        assert held and not accept(f, a)['ok'] and not reject(f, a)['ok']
    test_eq(len(git(f, 'worktree', 'list').splitlines()), 1)
    test_eq(pending(Path(d)/'none'), [])
    git(f, 'checkout', '-q', '--', '.'); reject(f, a)
    propose(f, {'AGENTS.md': 'A rule a person wrote.\n' + 'x' * LEARNED_MAX}, a)   # past Dhrona's limit
    head, r = git(f, 'rev-parse', 'HEAD'), accept(f, a)
    assert not r['ok'] and r['length'] > LEARNED_MAX and git(f, 'rev-parse', 'HEAD') == head, r
    assert a in learned_command(f)
    test_eq(learned_command(f, f'reject {a}'), f'rejected {a}')
    test_eq(learned_command(f), 'nothing is pending')

In [ ]:
with tempfile.TemporaryDirectory() as d:
    f = Path(d)/'learned'; f.mkdir()
    (f/'AGENTS.md').write_text('A rule a person wrote.\n')
    git(f, 'init', '-q'); git(f, 'add', '-A'); git(f, '-c', 'user.name=t', '-c', 'user.email=t@x', 'commit', '-qm', 'first')
    a = '20261019-010101-aaaaaa'
    propose(f, {'AGENTS.md': 'A rule a person wrote.\n\n' + 'Read sizes with ls -l. ' * 40 + '\n'}, a); accept(f, a)
    r = compact(f, chat=lambda prompt: '[kept 1]\n\nRead sizes with ls -l.')
    assert r['ok'] and r['before'] > r['after'] and r['id'] in learned_command(f), r
    test_eq(git(f, 'show', f"{PENDING}{r['id']}:AGENTS.md"), 'A rule a person wrote.\n\nRead sizes with ls -l.')
    assert not compact(f)['ok']                            # the pending compaction stands in the way
    import sys; mod = sys.modules.pop('dhrona.improve'); sys.modules['dhrona.improve'] = None   # dhrona not installed
    try: assert not compact(f)['ok']
    finally: sys.modules['dhrona.improve'] = mod

In [ ]:
with tempfile.TemporaryDirectory() as d:
    f = Path(d)/'learned'; f.mkdir()
    (f/'AGENTS.md').write_text('A rule a person wrote.\n')
    git(f, 'init', '-q'); git(f, 'add', '-A'); git(f, '-c', 'user.name=t', '-c', 'user.email=t@x', 'commit', '-qm', 'first')
    a, big, x, y = '20261019-010101-aaaaaa', '20261019-020202-bbbbbb', '20261019-030303-cccccc', '20261019-040404-dddddd'
    lessons = 'A rule a person wrote.\n\n' + 'Read sizes with ls -l. ' * 40 + '\n'
    propose(f, {'AGENTS.md': lessons}, a); accept(f, a)
    propose(f, {'AGENTS.md': lessons + '\n' + 'Run pytest -q before a commit. ' * 40 + '\n'}, big)
    propose(f, {'skills/rg/SKILL.md': '---\nname: rg\ndescription: x\n---\nuse rg\n'}, x)
    propose(f, {'AGENTS.md': lessons.replace('Read sizes', 'Read file sizes', 1)}, y)
    r = accept(f, big)
    assert not r['ok'] and r['length'] > LEARNED_MAX, r
    r = compact(f, big, chat=lambda prompt: '[kept 1]\n\nRead sizes with ls -l.\n\nRun pytest -q before a commit.')
    assert r['ok'] and r['before'] > LEARNED_MAX >= r['after'], r
    assert not compact(f, '20990101-000000-abcdef')['ok']
    r = accept(f, r['id'])                                  # the compaction carries `big` with it
    assert r['ok'] and (f/'AGENTS.md').read_text() == 'A rule a person wrote.\n\nRead sizes with ls -l.\n\nRun pytest -q before a commit.\n', r
    test_eq((r['moved'], r['stale']), ([x], [y]))           # `x` moved onto the compacted file; `y` edits a line it rewrote
    test_eq([(p['id'], p['stale']) for p in pending(f)], [(x, None), (y, STALE)])
    assert f'{y}  AGENTS.md' in learned_command(f)
    test_eq(git(f, 'rev-parse', f'{PENDING}{x}^'), git(f, 'rev-parse', 'HEAD'))
    assert accept(f, x)['ok'] and accept(f, y)['error'] == STALE

`learned_command` is the text form a terminal shows. `/learned` in a session and `ramabana-learned` from a shell both use it: with no argument it lists what is pending, `diff ID`, `accept ID` and `reject ID` act on one, and `compact` proposes a shorter `AGENTS.md`.

Learning is one setting per machine: `learning` in config.toml (see [setup](19_setup.ipynb)). It is on unless set off. `set_enabled(cfg, False)`, `/learned off` in a session and `ramabana-learned off` in a shell turn it off for every session on this machine, whatever its config folder. When learning is off:

- The briefing leaves out the learned `AGENTS.md` and the learned skills.
- The briefing drops its rule to record lessons.
- `remember` keeps plain notes only.
- `Agent.lessons()` returns none.
- The session writes no `start.json`.
- Dhrona's nightly schedule returns "learning is off" and does not touch the learned folder.

Lessons already kept stay where they are, unread. The review queue still works, and a person can clear it. `Agent(learned=False)` and the CLI's `--no-learned` turn learning off for one session only.

Ramabana 0.2.44 kept the switch in each config folder's `settings.json`. When config.toml does not set `learning`, `enabled(cfg)` reads the value from `<cfg>/settings.json` and writes it into config.toml. A choice made there survives. If the write fails, `enabled` keeps answering from `settings.json`. Ramabana never writes the `settings.json` that an embedding app shares. Ramabana never rewrites a config.toml that is not valid TOML. `set_enabled` raises `ValueError` saying so, and `/learned on` returns that message:

In [ ]:
with tempfile.TemporaryDirectory() as d:
    os.environ['RAMABANA_CONFIG'] = str(toml := Path(d)/'config.toml')
    try:
        assert enabled(d)
        test_eq(learned_command(learned_folder(d), 'off'), 'learning is off')
        test_eq((enabled(d), (Path(d)/SETTINGS).exists()), (False, False))
        assert '\nlearning = false\n' in toml.read_text()
        test_eq(set_enabled(d, True), True)
        toml.unlink(); (Path(d)/SETTINGS).write_text(json.dumps({'editor': {'tabSize': 2}, 'learning': 'off'}))
        test_eq(enabled(d), False)
        assert 'learning = false' in toml.read_text() and json.loads((Path(d)/SETTINGS).read_text())['learning'] == 'off'
        toml.write_text('learning = [')
        learned_command(learned_folder(d), 'on')
        test_eq(toml.read_text(), 'learning = [')
    finally: os.environ.pop('RAMABANA_CONFIG')

In [ ]:
#| export
@call_parse
def learned_cli(
    cmd:str='list',               # list, diff, accept, reject, compact, or on, off or status for learning itself
    id:str='',                    # the pending id, for diff, accept and reject
    cfg:str='~/.config/ramabana', # the config folder whose `learned` folder to review
):
    "Review what Dhrona proposed for the learned folder: list it, diff, accept or reject one, or ask for a compaction."
    print(learned_command(learned_folder(cfg), f'{cmd} {id}'.strip()))